# project_06_pdl1_binder — all notebooks (00→05) in one

This is a **convenience copy** that concatenates the six standalone notebooks in order so you can run the whole project top-to-bottom in a single Colab session. The individual notebooks (`00_setup.ipynb` … `05_validation_plan.ipynb`) remain in this folder and are the canonical deliverables. Sections are separated by dividers; each section keeps its own setup/`import` cells (re-running them is harmless). All synthetic numbers are still labeled `EXAMPLE_DATA`.

---

## ▶︎ Section 1 / 6 — `00_setup.ipynb`

---

# 00 · Environment Setup — De Novo Protein Design Capstone

This is the **shared setup notebook** every project starts from. Run it top to bottom
*once per Colab session*. It:

1. detects your GPU and warns if you're on a weak/absent one,
2. installs a light, pinned core toolset (Biopython, py3Dmol, foldseek-less utilities),
3. optionally installs heavier tools (ColabFold, ESMFold) on demand,
4. prints exact versions for your `LOG.md` (reproducibility is graded).

> **Compute reality.** A free Colab **T4** runs ColabFold, ESMFold, ProteinMPNN, and small
> RFdiffusion jobs. **BindCraft / RFantibody / large RFdiffusion** want an **A100** (Colab Pro+
> or a cluster). Each project's `MANUAL.md` states its tier. Don't fight a T4 to do an A100 job —
> plan your batch sizes around it.

## 1 · GPU & environment check

In [1]:
import subprocess, sys, platform, textwrap

def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()

print("Python :", sys.version.split()[0])
print("Platform:", platform.platform())

gpu = sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null")
if gpu:
    print("GPU    :", gpu)
    name = gpu.lower()
    if "t4" in name:
        print(textwrap.fill(
            "NOTE: T4 detected. Good for ColabFold/ESMFold/ProteinMPNN/small RFdiffusion. "
            "For BindCraft/RFantibody/large diffusion, switch to A100 (Colab Pro+) or a cluster.", 88))
    elif any(x in name for x in ("a100", "l4", "v100")):
        print("NOTE: capable GPU — heavier tools (BindCraft/RFantibody) are feasible.")
else:
    print("GPU    : NONE FOUND")
    print(textwrap.fill(
        "WARNING: No GPU. Go to Runtime → Change runtime type → Hardware accelerator → GPU. "
        "Structure prediction on CPU is impractically slow.", 88))

Python : 3.11.15
Platform: Linux-6.18.5-x86_64-with-glibc2.39
GPU    : NONE FOUND
Structure prediction on CPU is impractically slow.


## 2 · Pinned core install (fast, T4-friendly)

These are light and used across every project. Pins are conservative; bump them in your repo if needed and **log it**.

In [2]:
# Core utilities used in every project. Quiet + pinned.
%pip -q install biopython==1.84 py3Dmol==2.4.0 numpy pandas matplotlib seaborn tqdm requests 2>/dev/null
print("Core install done.")

Note: you may need to restart the kernel to use updated packages.
Core install done.


In [3]:
# Version stamp — copy this block's output into your LOG.md for reproducibility.
import importlib, datetime
mods = ["Bio", "py3Dmol", "numpy", "pandas", "matplotlib", "seaborn", "tqdm", "requests"]
print("# Environment stamp", datetime.datetime.utcnow().isoformat(timespec="seconds"), "UTC")
for m in mods:
    try:
        v = importlib.import_module(m).__version__
    except Exception:
        v = "n/a"
    print(f"{m:14s} {v}")

# Environment stamp 2026-06-24T03:15:01 UTC


Bio            1.84
py3Dmol        2.4.0


numpy          2.4.6


pandas         3.0.3
matplotlib     3.11.0


seaborn        0.13.2
tqdm           4.68.3
requests       2.33.1


## 3 · Heavy tools — install *on demand*

Don't install these unless your project needs them this session (they're slow to set up).
Each is wrapped in a function so you only pay the cost when you call it.

In [4]:
def install_colabfold():
    """ColabFold (AF2). ~3–5 min on first install. T4 OK."""
    import subprocess
    subprocess.run("pip -q install 'colabfold[alphafold-minus-jax]'", shell=True)
    # On Colab, the standard route is the localcolabfold installer or the ColabFold notebook;
    # here we expose the pip route. If it fails, fall back to the official ColabFold notebook
    # and import your sequences. Log whichever path you used.
    print("ColabFold install attempted. Verify with: from colabfold.batch import run")

def install_esmfold():
    """ESMFold via HuggingFace transformers. T4 OK for <~400 aa."""
    import subprocess
    subprocess.run("pip -q install 'transformers>=4.40' accelerate", shell=True)
    print("ESMFold deps installed. Load with transformers EsmForProteinFolding.")

print("Helpers ready: install_colabfold(), install_esmfold().")

Helpers ready: install_colabfold(), install_esmfold().


## 4 · Reproducibility helpers

Call `set_seeds()` at the top of every run, and use `log()` to append to your `LOG.md`.

In [5]:
import os, random
import numpy as np

def set_seeds(seed: int = 0):
    random.seed(seed); np.random.seed(seed); os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass
    print(f"seeds set to {seed}")

def log(msg: str, path: str = "LOG.md"):
    import datetime
    stamp = datetime.datetime.utcnow().isoformat(timespec="seconds")
    with open(path, "a") as fh:
        fh.write(f"- {stamp}Z · {msg}\n")
    print("logged:", msg)

set_seeds(0)
log("Ran 00_setup; environment stamped.")

seeds set to 0
logged: Ran 00_setup; environment stamped.


## 5 · (Optional) Mount Google Drive for persistence

Colab sessions are ephemeral. Mount Drive to keep your `results/` and design pools between sessions.

In [6]:
# from google.colab import drive
# drive.mount("/content/drive")
# WORKDIR = "/content/drive/MyDrive/denovo_capstone/project_XX"
# import os; os.makedirs(WORKDIR, exist_ok=True); os.chdir(WORKDIR)
print("Uncomment to mount Drive and set your working directory.")

Uncomment to mount Drive and set your working directory.


---
**Next:** open `01_define_and_explore.ipynb`. Keep this session alive — re-running `00_setup`
each new session is normal. Record every version and seed in `LOG.md`.

---

## ▶︎ Section 2 / 6 — `01_define_and_explore.ipynb`

---

# 01 · Define & Explore — target prep, PD-1-face hotspots, binder metrics

**Standard slot:** *define & explore.* **For Project 06 this means:** clean the PD-L1 ectodomain,
select the **hotspot residues on the PD-1-binding (competitive) face**, write down the binder metrics
+ cutoffs, and run a deterministic **mock** mini-run as your "hello-world" (D0).

Run `00_setup.ipynb` first in this session. A real binder campaign wants an **A100** (see
`MANUAL.md §2`); everything here runs on a no-GPU **mock** backend so you can build the plumbing
anywhere, then switch to the real backend on Colab Pro / A100.

## The binder metrics, precisely

| Metric | Range | Means | Does **not** mean |
|--------|-------|-------|-------------------|
| pLDDT | 0–100 | per-residue *local* confidence of the binder | thermostability / ΔG |
| **pae_interaction** | Å | AF2-Multimer error across the **binder–target interface** (the key binder metric) | measured affinity |
| scRMSD | Å | designed-vs-predicted Cα-RMSD (self-consistency) | binding/function |
| rosetta_dG | REU | interface energy (more negative = stronger) | a guarantee it binds |
| shape complementarity | 0–1 | interface packing quality | epitope correctness |
| TM-score | 0–1 | similarity to nearest known fold (<0.5 ≈ novel) | a pass/fail of correctness |

The shared `"binder"` cutoffs: **scRMSD ≤ 2.5, pLDDT ≥ 80, pae_interaction ≤ 10, rosetta_dG ≤ −30,
sc ≥ 0.6.** `pae_interaction` is the single most important binder metric — but a low value is
*confidence*, **not** affinity. A passing design is a **hypothesis** until SPR/BLI.

## Setup paths

In [7]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

paths ready; cwd = /home/user/biofx_python/denovo_protein_design_course/projects/project_06_pdl1_binder/notebooks


## 1 · Target prep + PD-1-face hotspots

The design target is the **PD-L1 ectodomain (IgV domain)**, and the hotspots are the PD-L1 residues
that **PD-1 contacts** in the PD-1/PD-L1 complex — steering the binder there is what makes it a
*competitive blocker*. Fetch the candidate complexes with `data/download_data.py` (4ZQK / 5O45 —
**verify on RCSB**), isolate the PD-L1 chain, remove PD-1/waters/heteroatoms, and read the interface
residues off the complex.

Below we just *declare* an EXAMPLE hotspot set so the notebook runs end-to-end; **replace it with the
residues you derive from the actual PD-1/PD-L1 interface** (numbering depends on the PDB you verify).

In [8]:
import binder_tools as bt

TARGET = "PDL1"                      # cleaned PD-L1 IgV domain (you produce this from 4ZQK/5O45)
# EXAMPLE hotspots on the PD-1-binding face — VERIFY/REPLACE from the PD-1/PD-L1 interface (data/README.md).
# These are placeholders so the plumbing runs; real numbering depends on the PDB chain you clean.
HOTSPOTS = bt.parse_hotspots("A56,A66,A115")   # EXAMPLE_DATA placeholder residues
print("target  :", TARGET)
print("hotspots:", HOTSPOTS, " (EXAMPLE — replace with your verified PD-1-face residues)")

target  : PDL1
hotspots: ('A115', 'A56', 'A66')  (EXAMPLE — replace with your verified PD-1-face residues)


## 2 · Mock hello-world: a tiny two-paradigm mini-run

`scripts/binder_tools.py` exposes both paradigms behind one API:
`generate_binders_bindcraft(...)` and `generate_binders_rfdiffusion(...)`, plus `af2_multimer(...)`
(the scorer). The **mock** backend is deterministic and GPU-free so you can develop the plumbing.
**Never report mock numbers as real** — they are `SYNTHETIC` by construction.

In [9]:
# A few designs from each paradigm, scored by mock AF2-Multimer. All numbers are SYNTHETIC.
bc = bt.generate_binders_bindcraft(TARGET, HOTSPOTS, n=3, tool="mock")
rf = bt.generate_binders_rfdiffusion(TARGET, HOTSPOTS, n=3, tool="mock")
bt.score_designs(bc, tool="mock")
bt.score_designs(rf, tool="mock")

d = bc[0]
print("example BindCraft design:")
print("  id   :", d.design_id)
print("  len  :", d.length, "aa")
print("  seq  :", d.sequence)
print("  pae_interaction =", d.pae_interaction, " scrmsd =", d.scrmsd,
      " sc =", d.shape_complementarity, " (SYNTHETIC)")
print("  synthetic flag  :", d.synthetic, "->", d.notes[0])
print("\nReminder: switch tool='mock' -> 'bindcraft'/'rfdiffusion'/'af2' on Colab (A100). See MANUAL.md §2.")

example BindCraft design:
  id   : EXAMPLE_DATA_bindcraft_0000
  len  : 71 aa
  seq  : DYTCMIFLHITGDNPQWIALMYALHYPLHEFCHYKCMYFLRNTVHYACDIKLDYFQMEAGRSACDIKVRYF
  pae_interaction = 19.0  scrmsd = 1.21  sc = 0.81  (SYNTHETIC)
  synthetic flag  : True -> SYNTHETIC — mock backend, not a real design/prediction

Reminder: switch tool='mock' -> 'bindcraft'/'rfdiffusion'/'af2' on Colab (A100). See MANUAL.md §2.


## 3 · Epitope-competition proxy (does it cover the PD-1 footprint?)

A binder only *blocks* PD-1 if it overlaps the PD-1 footprint enough. `hotspot_overlap()` is a
geometry proxy (fraction of hotspots contacted) — a teaching stand-in for the PD-1-competition assay
in notebook 04. Higher ⇒ more likely to block (not a guarantee).

In [10]:
for b in bc[:3]:
    ov = bt.hotspot_overlap(b.contact_residues, HOTSPOTS)
    print(f"{b.design_id}: contacts {b.contact_residues} -> PD-1-footprint overlap = {ov} (SYNTHETIC)")

EXAMPLE_DATA_bindcraft_0000: contacts ('A115',) -> PD-1-footprint overlap = 0.333 (SYNTHETIC)
EXAMPLE_DATA_bindcraft_0001: contacts ('A115', 'A56') -> PD-1-footprint overlap = 0.667 (SYNTHETIC)
EXAMPLE_DATA_bindcraft_0002: contacts ('A115',) -> PD-1-footprint overlap = 0.333 (SYNTHETIC)


## Visualize a binder–target complex (py3Dmol)

Use this to eyeball a predicted binder–PD-L1 complex once you have a real PDB (from AF2-Multimer).

In [11]:
import py3Dmol

def show_complex(pdb_path_or_str, is_path=True):
    data = open(pdb_path_or_str).read() if is_path else pdb_path_or_str
    view = py3Dmol.view(width=520, height=420)
    view.addModel(data, "pdb")
    view.setStyle({"cartoon": {"color": "spectrum"}})
    view.zoomTo()
    return view.show()

# Example (after a real AF2-Multimer prediction writes a complex PDB):
# show_complex("results/af2/top_complex.pdb")
print("show_complex(pdb_path) ready.")

show_complex(pdb_path) ready.


## D0 checklist
- [ ] PD-1/PD-L1 accessions verified on RCSB (4ZQK/5O45 are candidates); PD-L1 chain + IgV domain identified.
- [ ] Cleaned PD-L1 target + **PD-1-face hotspot list** (derived from the interface, not invented).
- [ ] One-paragraph definition of each binder metric **with** its "does not mean" note.
- [ ] Reproduced mock mini-run (both paradigms) with metrics printed and flagged SYNTHETIC.
- [ ] Problem statement with measurable success criteria + controls; `LOG.md` entry (GPU, seed).

**Next:** `02_generate.ipynb` — the two-paradigm binder campaign.

---

## ▶︎ Section 3 / 6 — `02_generate.ipynb`

---

# 02 · Campaign — two-paradigm binder design vs PD-L1

**Standard slot:** *design campaign.* **For Project 06 this is the core:** run **both** paradigms
against the PD-1-face hotspots and assemble their pools (D2):
- **BindCraft** (one-shot hallucination, AF2-Multimer in the loop) — **50–200** designs.
- **RFdiffusion binder mode → ProteinMPNN** — **500–1000** backbones → sequences.

Then score every design with **AF2-Multimer** (`pae_interaction` is the key binder metric).

> **Compute honesty:** a real campaign at this scale wants an **A100** (Colab Pro+ or a cluster).
> Free **T4** = a *small fallback* (FreeBindCraft, small `num_designs`, a small RFdiffusion batch +
> ESMFold triage). The cells below run on the deterministic **mock** backend so the plumbing executes
> anywhere; the real calls + A100 notes are shown alongside. Run `00_setup.ipynb` first.

## Setup paths

In [12]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

paths ready; cwd = /home/user/biofx_python/denovo_protein_design_course/projects/project_06_pdl1_binder/notebooks


## Version-verify the pinned upstreams (tools change!)

The binder tools live in fast-moving upstream repos. **Pin commits** and **verify the URLs still
exist** before relying on them (`requests.head`; a non-200 means it moved — update the pin and log
it). The generation itself needs an A100; this check needs nothing.

In [13]:
import requests

# Pinned upstreams (pin a COMMIT/tag in your repo — these change; commit hashes go in comments):
#   BindCraft      https://github.com/martinpacesa/BindCraft        # e.g. pin <commit>
#   FreeBindCraft  https://github.com/cytokineking/FreeBindCraft     # free-tier fallback — VERIFY it exists; pin <commit>
#   RFdiffusion    https://github.com/RosettaCommons/RFdiffusion     # pin <commit>
#   ColabDesign    https://github.com/sokrypton/ColabDesign          # RFdiffusion-binder + ProteinMPNN; pin <commit>
#   ColabFold      https://github.com/sokrypton/ColabFold            # AF2-Multimer; pin <commit>
PINNED = {
    "BindCraft":     "https://github.com/martinpacesa/BindCraft",
    "FreeBindCraft": "https://github.com/cytokineking/FreeBindCraft",
    "RFdiffusion":   "https://github.com/RosettaCommons/RFdiffusion",
    "ColabDesign":   "https://github.com/sokrypton/ColabDesign",
    "ColabFold":     "https://github.com/sokrypton/ColabFold",
}
for name, url in PINNED.items():
    try:
        r = requests.head(url, allow_redirects=True, timeout=15)
        print(f"  [{r.status_code}] {name:14s} {url}")
    except Exception as e:  # noqa: BLE001
        print(f"  [ERR] {name:14s} {url}  ({e})")
print("\nNon-200 / error => the upstream moved; update the pin in env/requirements.txt and log it.")
print("FreeBindCraft especially: VERIFY it still exists before relying on the free-tier fallback.")

  [200] BindCraft      https://github.com/martinpacesa/BindCraft


  [200] FreeBindCraft  https://github.com/cytokineking/FreeBindCraft


  [200] RFdiffusion    https://github.com/RosettaCommons/RFdiffusion


  [200] ColabDesign    https://github.com/sokrypton/ColabDesign


  [200] ColabFold      https://github.com/sokrypton/ColabFold

Non-200 / error => the upstream moved; update the pin in env/requirements.txt and log it.
FreeBindCraft especially: VERIFY it still exists before relying on the free-tier fallback.


## 1 · Define the campaign

Same target + hotspots as notebook 01. Set honest campaign sizes; the cells run on `mock` so they
execute anywhere. On Colab (A100) switch `TOOL_*` to the real backends — and **shrink the numbers on
a T4** (FreeBindCraft, a small RFdiffusion batch).

In [14]:
import binder_tools as bt
import pandas as pd

TARGET = "PDL1"
HOTSPOTS = bt.parse_hotspots("A56,A66,A115")   # EXAMPLE — replace with your verified PD-1-face residues

# Honest campaign sizes (catalog): BindCraft 50-200, RFdiffusion 500-1000 backbones.
# We use small mock counts here so the dry run is fast; scale up with the real backend on A100.
N_BINDCRAFT   = 60      # -> 50-200 on A100; fewer (FreeBindCraft) on T4
N_RFDIFFUSION = 200     # -> 500-1000 backbones on A100; small batch on T4

TOOL_BINDCRAFT   = "mock"   # -> "bindcraft" / "freebindcraft" on Colab
TOOL_RFDIFFUSION = "mock"   # -> "rfdiffusion" on Colab
TOOL_AF2         = "mock"   # -> "af2" (ColabFold AF2-Multimer) on Colab

print(f"BindCraft   : n={N_BINDCRAFT}  tool={TOOL_BINDCRAFT}")
print(f"RFdiffusion : n={N_RFDIFFUSION} tool={TOOL_RFDIFFUSION}")
print(f"AF2-Multimer: tool={TOOL_AF2}")
print("hotspots    :", HOTSPOTS)

BindCraft   : n=60  tool=mock
RFdiffusion : n=200 tool=mock
AF2-Multimer: tool=mock
hotspots    : ('A115', 'A56', 'A66')


## 2 · Paradigm #1 — BindCraft campaign

One-shot hallucination with AF2-Multimer in the loop. On A100 this produces 50–200 binders
pre-filtered on interface confidence; we still re-score with AF2-Multimer so the head-to-head with
RFdiffusion is apples-to-apples. The `mock` backend returns deterministic `SYNTHETIC` designs.

In [15]:
# Real call (Colab, A100): bt.generate_binders_bindcraft(TARGET, HOTSPOTS, n=N_BINDCRAFT, tool="bindcraft")
#   free-tier fallback: tool="freebindcraft", smaller N. See MANUAL.md §2 / scripts/binder_tools.py TODOs.
bindcraft = bt.generate_binders_bindcraft(TARGET, HOTSPOTS, n=N_BINDCRAFT, tool=TOOL_BINDCRAFT)
bt.score_designs(bindcraft, tool=TOOL_AF2)     # AF2-Multimer -> pae_interaction, plddt, scrmsd, sc
print(f"BindCraft pool: {len(bindcraft)} designs (tool={TOOL_BINDCRAFT}; SYNTHETIC if mock)")
print("example:", bindcraft[0].design_id, "pae_interaction=", bindcraft[0].pae_interaction)

BindCraft pool: 60 designs (tool=mock; SYNTHETIC if mock)
example: EXAMPLE_DATA_bindcraft_0000 pae_interaction= 19.0


## 3 · Paradigm #2 — RFdiffusion binder campaign → ProteinMPNN

Diffuse binder backbones docked at the hotspots, then ProteinMPNN designs sequences, then
AF2-Multimer re-predicts each complex. On A100 this is 500–1000 backbones (the per-backbone hit rate
is low — that is normal). The `mock` backend stands in for the whole chain.

In [16]:
# Real call (Colab, A100): bt.generate_binders_rfdiffusion(TARGET, HOTSPOTS, n=N_RFDIFFUSION,
#   tool="rfdiffusion", mpnn_temperature=0.1, num_seq_per_backbone=8). AF2-Multimer is the slow step.
rfdiff = bt.generate_binders_rfdiffusion(TARGET, HOTSPOTS, n=N_RFDIFFUSION, tool=TOOL_RFDIFFUSION)
bt.score_designs(rfdiff, tool=TOOL_AF2)
print(f"RFdiffusion pool: {len(rfdiff)} designs (tool={TOOL_RFDIFFUSION}; SYNTHETIC if mock)")
print("example:", rfdiff[0].design_id, "pae_interaction=", rfdiff[0].pae_interaction)

RFdiffusion pool: 200 designs (tool=mock; SYNTHETIC if mock)
example: EXAMPLE_DATA_rfdiffusion_0000 pae_interaction= 7.0


## 4 · Assemble + persist both pools

Write one tidy CSV per paradigm (plus a combined one). These feed notebook 03 (the shared filter).
We add an EXAMPLE physics column (`rosetta_dG`) here so the binder physics layer has something to act
on in the dry run — on Colab these come from FreeBindCraft/PyRosetta; for `mock` they are SYNTHETIC.

In [17]:
import pandas as pd

def pool_to_df(designs):
    rows = []
    for d in designs:
        # In the mock dry run we attach an EXAMPLE_DATA interface energy so Layer 3 (physics) is
        # exercised. On Colab, replace with the real FreeBindCraft/PyRosetta rosetta_dG + solubility.
        rdg = -45.0 + (bt._hashints("dG", d.design_id) % 40)   # SYNTHETIC, range ~ -45..-6 REU
        rows.append(dict(
            design_id=d.design_id, paradigm=d.paradigm, target=d.target,
            length=d.length, sequence=d.sequence,
            plddt=d.plddt, pae_interaction=d.pae_interaction, scrmsd=d.scrmsd,
            shape_complementarity=d.shape_complementarity,
            rosetta_dG=round(float(rdg), 2), solubility=0.3,
            contact_residues=",".join(d.contact_residues),
            hotspot_overlap=bt.hotspot_overlap(d.contact_residues, d.hotspots),
            synthetic=d.synthetic,
        ))
    return pd.DataFrame(rows)

df_bc = pool_to_df(bindcraft); df_bc.to_csv("results/bindcraft_designs.csv", index=False)
df_rf = pool_to_df(rfdiff);    df_rf.to_csv("results/rfdiffusion_designs.csv", index=False)
combined = pd.concat([df_bc, df_rf], ignore_index=True)
combined.to_csv("results/all_designs.csv", index=False)

print("wrote results/bindcraft_designs.csv   ", df_bc.shape)
print("wrote results/rfdiffusion_designs.csv ", df_rf.shape)
print("wrote results/all_designs.csv         ", combined.shape)
print("\nALL numbers are SYNTHETIC in the mock dry run (EXAMPLE_DATA) — never report as real results.")
combined.head(4)

wrote results/bindcraft_designs.csv    (60, 14)
wrote results/rfdiffusion_designs.csv  (200, 14)
wrote results/all_designs.csv          (260, 14)

ALL numbers are SYNTHETIC in the mock dry run (EXAMPLE_DATA) — never report as real results.


,design_id,paradigm,target,length,sequence,plddt,pae_interaction,scrmsd,shape_complementarity,rosetta_dG,solubility,contact_residues,hotspot_overlap,synthetic
0,EXAMPLE_DATA_bindcraft_0000,bindcraft,PDL1,71,DYTCMIFLHITGDNPQWIALMYALHYPLHEFCHYKCMYFLRNTVHY...,91.0,19.0,1.21,0.81,-38.0,0.3,A115,0.333,True
1,EXAMPLE_DATA_bindcraft_0001,bindcraft,PDL1,58,LDYPLWIAQDEFQDSKCMSKCRNACDSKGDSTLRNTLRIFVRIFLR...,97.0,7.0,4.17,0.87,-19.0,0.3,"A115,A56",0.667,True
2,EXAMPLE_DATA_bindcraft_0002,bindcraft,PDL1,65,RNKQWIKVRIPQHYAVHNKCMEFGRITQWSACRSFGHIAVDNPQME...,97.0,19.0,1.17,0.57,-35.0,0.3,A115,0.333,True
3,EXAMPLE_DATA_bindcraft_0003,bindcraft,PDL1,76,TVDEPLDIAGRYAQMIFVREFQRNAVDIKCHSALDYTLMETLMIPQ...,90.0,16.0,2.10,0.65,-38.0,0.3,"A115,A56",0.667,True


## D2 checklist
- [ ] BindCraft pool generated at honest scale (50–200 on A100; FreeBindCraft/small on T4).
- [ ] RFdiffusion-binder pool generated (500–1000 backbones → ProteinMPNN on A100).
- [ ] Every design scored by AF2-Multimer (`pae_interaction` parsed); both pools written to `results/`.
- [ ] Design log: every config + seed + tool **commit** + output path, in `LOG.md`.
- [ ] Version-verify output captured; 3–4 page interim report.

**Next:** `03_filter_and_rank.ipynb` — run the **shared** filter on both pools.

---

## ▶︎ Section 4 / 6 — `03_filter_and_rank.ipynb`

---

# 03 · Filter & Rank — run the shared multi-layer filter (binder cutoffs)

**Standard slot:** *filter & rank* via `shared/filtering_pipeline.py` — the same module all 25
projects use. **For Project 06** you build `fp.Design` **binder** objects from both pools, call
`fp.run_pipeline(..., design_type="binder")`, and `fp.report(...)` the survival funnel + ranked CSV,
**per paradigm** so the head-to-head is fair (D3 part 1).

> **Do not fork the module into this project.** Iterate against `shared/filtering_pipeline.py` and PR
> improvements back. This notebook *imports* it.

Run `00`–`02` first so `results/bindcraft_designs.csv` + `results/rfdiffusion_designs.csv` exist.

## Setup paths

In [18]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

paths ready; cwd = /home/user/biofx_python/denovo_protein_design_course/projects/project_06_pdl1_binder/notebooks


## Load the shared filtering pipeline
This is the cohort's shared module — improvements here are pull-requested back for everyone. Note the
`"binder"` cutoffs: scRMSD ≤ 2.5, pLDDT ≥ 80, pae ≤ 10, rosetta_dG ≤ −30, sc ≥ 0.6.

In [19]:
import filtering_pipeline as fp
import pandas as pd

print("Loaded shared filtering_pipeline from:", fp.__file__)
print("binder cutoffs:", fp.DEFAULT_CUTOFFS["binder"])

Loaded shared filtering_pipeline from: /home/user/biofx_python/denovo_protein_design_course/shared/filtering_pipeline.py
binder cutoffs: {'scrmsd': 2.5, 'plddt': 80, 'pae': 10, 'rosetta_dG': -30, 'sc': 0.6}


## Build `Design` (binder) objects from the pools

Map each pool row onto `fp.Design` with `design_type="binder"`. The binder metrics drive the layers:
`scrmsd`/`plddt`/`pae_interaction` (Layer 1 self-consistency), and `rosetta_dG`/`shape_complementarity`/`solubility`
(Layer 3 physics). We keep `paradigm` + `hotspot_overlap` in `extra` for the head-to-head + epitope
analysis in notebook 04. (Mock has no independent orthogonal predictor, so we run Layers 1+3 here;
on Colab add a second predictor for Layer 2.)

In [20]:
import os
import pandas as pd

# Regenerate the pools if a fresh session lost them (deterministic mock).
if not (os.path.exists("results/bindcraft_designs.csv") and os.path.exists("results/rfdiffusion_designs.csv")):
    import binder_tools as bt
    TARGET, HOTSPOTS = "PDL1", bt.parse_hotspots("A56,A66,A115")
    bc = bt.generate_binders_bindcraft(TARGET, HOTSPOTS, n=60, tool="mock");  bt.score_designs(bc, tool="mock")
    rf = bt.generate_binders_rfdiffusion(TARGET, HOTSPOTS, n=200, tool="mock"); bt.score_designs(rf, tool="mock")
    def _q(designs, p):
        rows=[dict(design_id=d.design_id, paradigm=d.paradigm, length=d.length, sequence=d.sequence,
                   plddt=d.plddt, pae_interaction=d.pae_interaction, scrmsd=d.scrmsd,
                   shape_complementarity=d.shape_complementarity,
                   rosetta_dG=round(-45.0+(bt._hashints("dG",d.design_id)%40),2), solubility=0.3,
                   hotspot_overlap=bt.hotspot_overlap(d.contact_residues,d.hotspots), synthetic=d.synthetic)
              for d in designs]
        pd.DataFrame(rows).to_csv(p, index=False)
    _q(bc, "results/bindcraft_designs.csv"); _q(rf, "results/rfdiffusion_designs.csv")

df_bc = pd.read_csv("results/bindcraft_designs.csv")
df_rf = pd.read_csv("results/rfdiffusion_designs.csv")

def row_to_binder(r):
    return fp.Design(
        design_id=str(r["design_id"]), sequence=str(r.get("sequence", "")), design_type="binder",
        plddt=r.get("plddt"), pae_interaction=r.get("pae_interaction"), scrmsd=r.get("scrmsd"),
        scrmsd_orthogonal=r.get("scrmsd"),   # mock: reuse scrmsd as a stand-in; use a 2nd predictor on Colab
        rosetta_dG=r.get("rosetta_dG"), shape_complementarity=r.get("shape_complementarity"),
        solubility=r.get("solubility", 0.3),
        extra={"paradigm": r.get("paradigm"), "hotspot_overlap": r.get("hotspot_overlap")},
    )

binders_bc = [row_to_binder(r) for _, r in df_bc.iterrows()]
binders_rf = [row_to_binder(r) for _, r in df_rf.iterrows()]
print(f"built {len(binders_bc)} BindCraft + {len(binders_rf)} RFdiffusion binder Designs")

built 60 BindCraft + 200 RFdiffusion binder Designs


## Run the pipeline — per paradigm (fair head-to-head)

`run_pipeline(design_type="binder")` applies the binder cutoffs in order and returns a ranked
DataFrame with survival counts in `df.attrs`. We run **each paradigm separately** so the
survival-at-each-layer funnels are comparable. We use Layers 1+3 here (mock has no independent
orthogonal source; add Layer 2 on Colab with a second predictor).

In [21]:
def run_one(designs, label):
    df = fp.run_pipeline(designs, design_type="binder", use_layers=(1, 3))
    df["paradigm"] = label
    surv = df.attrs["survival"]; n = df.attrs["n_total"]
    passed = int((df["layers_passed"] >= 3).sum())
    print(f"{label:12s}: {n} designs, survival {surv}, all-layers hit rate = {passed}/{n} ({100*passed/max(n,1):.1f}%)")
    return df

ranked_bc = run_one(binders_bc, "bindcraft")
ranked_rf = run_one(binders_rf, "rfdiffusion")

ranked = pd.concat([ranked_bc, ranked_rf], ignore_index=True).sort_values(
    ["layers_passed", "score"], ascending=False).reset_index(drop=True)
ranked.to_csv("results/all_ranked.csv", index=False)
print("\nwrote results/all_ranked.csv", ranked.shape)
ranked.head(10)[["design_id", "paradigm", "layers_passed", "score",
                 "scrmsd", "plddt", "pae_interaction", "rosetta_dG"]]

bindcraft   : 60 designs, survival {'L1': 11, 'L3': 3}, all-layers hit rate = 3/60 (5.0%)
rfdiffusion : 200 designs, survival {'L1': 33, 'L3': 11}, all-layers hit rate = 11/200 (5.5%)

wrote results/all_ranked.csv (260, 21)


,design_id,paradigm,layers_passed,score,scrmsd,plddt,pae_interaction,rosetta_dG
0,EXAMPLE_DATA_rfdiffusion_0176,rfdiffusion,3,4.2900,1.04,84.0,6.0,-37.0
1,EXAMPLE_DATA_bindcraft_0034,bindcraft,3,4.2600,0.87,97.0,9.0,-33.0
2,EXAMPLE_DATA_rfdiffusion_0004,rfdiffusion,3,4.1233,1.00,80.0,8.0,-37.0
3,EXAMPLE_DATA_rfdiffusion_0008,rfdiffusion,3,3.7933,1.87,97.0,5.0,-43.0
4,EXAMPLE_DATA_rfdiffusion_0131,rfdiffusion,3,3.6633,1.50,80.0,8.0,-39.0
5,EXAMPLE_DATA_rfdiffusion_0199,rfdiffusion,3,3.5967,1.42,92.0,10.0,-34.0
6,EXAMPLE_DATA_rfdiffusion_0157,rfdiffusion,3,3.5867,1.73,83.0,7.0,-41.0
7,EXAMPLE_DATA_rfdiffusion_0072,rfdiffusion,3,3.4167,2.24,94.0,4.0,-40.0
8,EXAMPLE_DATA_bindcraft_0048,bindcraft,3,3.3367,2.16,86.0,4.0,-36.0
9,EXAMPLE_DATA_bindcraft_0041,bindcraft,3,3.3000,1.67,87.0,9.0,-30.0


## Survival-at-each-layer via `report()`

`report()` prints the hit-rate accounting and draws the survival funnel. Here we report the **combined**
pool for one comparable figure; the per-paradigm runs above are the rigorous version. Read the bars as
a funnel: steep drops show which layer discriminates.

In [22]:
import matplotlib
matplotlib.use("Agg")  # headless-safe; Colab still displays inline

all_binders = binders_bc + binders_rf
df_all = fp.run_pipeline(all_binders, design_type="binder", use_layers=(1, 3))
top = fp.report(df_all, top_n=15, save_prefix="results/p06")
print("\nsaved results/p06_survival.png + results/p06_ranked.csv")
top

Total designs: 260
  L1 survivors: 44  (16.9%)
  L3 survivors: 14  (5.4%)



saved results/p06_survival.png + results/p06_ranked.csv


,design_id,design_type,layers_passed,score,scrmsd,plddt,pae_interaction,rosetta_dG,tm_to_pdb
0,EXAMPLE_DATA_rfdiffusion_0176,binder,3,4.2900,1.04,84.0,6.0,-37.0,None
1,EXAMPLE_DATA_bindcraft_0034,binder,3,4.2600,0.87,97.0,9.0,-33.0,None
2,EXAMPLE_DATA_rfdiffusion_0004,binder,3,4.1233,1.00,80.0,8.0,-37.0,None
3,EXAMPLE_DATA_rfdiffusion_0008,binder,3,3.7933,1.87,97.0,5.0,-43.0,None
4,EXAMPLE_DATA_rfdiffusion_0131,binder,3,3.6633,1.50,80.0,8.0,-39.0,None
5,EXAMPLE_DATA_rfdiffusion_0199,binder,3,3.5967,1.42,92.0,10.0,-34.0,None
6,EXAMPLE_DATA_rfdiffusion_0157,binder,3,3.5867,1.73,83.0,7.0,-41.0,None
7,EXAMPLE_DATA_rfdiffusion_0072,binder,3,3.4167,2.24,94.0,4.0,-40.0,None
8,EXAMPLE_DATA_bindcraft_0048,binder,3,3.3367,2.16,86.0,4.0,-36.0,None
9,EXAMPLE_DATA_bindcraft_0041,binder,3,3.3000,1.67,87.0,9.0,-30.0,None


## Honest hit-rate accounting (per paradigm)

Report `N passing all layers / N generated` for **each** paradigm — this is the number the head-to-head
in notebook 04 builds on. Remember: survival is *enrichment*, not *correctness*. Mock numbers are
SYNTHETIC.

In [23]:
for label, df in [("bindcraft", ranked_bc), ("rfdiffusion", ranked_rf)]:
    n = len(df); passed = int((df["layers_passed"] >= 3).sum())
    print(f"{label:12s}: layers_passed distribution {df['layers_passed'].value_counts().sort_index().to_dict()}")
    print(f"{'':12s}  all-layers survivors = {passed}/{n} ({100*passed/max(n,1):.1f}%)  [SYNTHETIC if mock]")

bindcraft   : layers_passed distribution {0: 49, 1: 8, 3: 3}
              all-layers survivors = 3/60 (5.0%)  [SYNTHETIC if mock]
rfdiffusion : layers_passed distribution {0: 167, 1: 22, 3: 11}
              all-layers survivors = 11/200 (5.5%)  [SYNTHETIC if mock]


## D3 (part 1) checklist
- [ ] `results/all_ranked.csv` produced by the **shared** module (`design_type="binder"`), not a one-off script.
- [ ] Survival-at-each-layer reported **per paradigm** (funnel figure `results/p06_survival.png`).
- [ ] Honest hit-rate accounting (N pass / N generated) for BindCraft and RFdiffusion.
- [ ] Mapping assumptions (which fields → which `Design` attributes) written down.

**Next:** `04_validate.ipynb` — the BindCraft-vs-RFdiffusion benchmark + epitope competition.

---

## ▶︎ Section 5 / 6 — `04_validate.ipynb`

---

# 04 · Validate — BindCraft vs RFdiffusion + epitope competition

**Standard slot:** *validate (in silico).* **For Project 06 this is the core comparison:** the
head-to-head between the two paradigms (hit rate, interface energy, novelty) plus **epitope-competition
reasoning vs PD-1**, with publication-style figures (D3 part 2).

Needs `results/bindcraft_designs.csv` + `results/rfdiffusion_designs.csv` + `results/all_ranked.csv`
(from notebooks 02–03).

## Setup paths

In [24]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

paths ready; cwd = /home/user/biofx_python/denovo_protein_design_course/projects/project_06_pdl1_binder/notebooks


## 1 · Head-to-head hit rate + interface energy

Compare the two paradigms on (a) all-layers **hit rate** and (b) the **interface-energy** (`rosetta_dG`)
distribution of survivors. A fair comparison filters both identically (notebook 03) and reports the
*distribution*, not the single best. Mock numbers are SYNTHETIC.

In [25]:
import pandas as pd, numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ranked = pd.read_csv("results/all_ranked.csv")
print("paradigms:", ranked["paradigm"].value_counts().to_dict())

summary = []
for p, g in ranked.groupby("paradigm"):
    n = len(g); passed = int((g["layers_passed"] >= 3).sum())
    summary.append(dict(paradigm=p, n=n, all_layers_survivors=passed,
                        hit_rate_pct=round(100*passed/max(n,1), 1),
                        median_pae=round(float(g["pae_interaction"].median()), 2),
                        median_dG=round(float(g["rosetta_dG"].median()), 2)))
summary = pd.DataFrame(summary)
print("\nhead-to-head summary (SYNTHETIC if mock):")
print(summary.to_string(index=False))

paradigms: {'rfdiffusion': 200, 'bindcraft': 60}

head-to-head summary (SYNTHETIC if mock):
   paradigm   n  all_layers_survivors  hit_rate_pct  median_pae  median_dG
  bindcraft  60                     3           5.0        10.5      -31.0
rfdiffusion 200                    11           5.5        12.0      -26.5


In [26]:
# Interface-energy distribution per paradigm (survivors).
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
for p, g in ranked.groupby("paradigm"):
    surv = g[g["layers_passed"] >= 3]
    ax[0].hist(g["pae_interaction"].dropna(), bins=15, alpha=0.5, label=p)
    ax[1].hist(surv["rosetta_dG"].dropna(), bins=15, alpha=0.5, label=p)
ax[0].set_xlabel("pae_interaction (Å, lower better)"); ax[0].set_ylabel("designs"); ax[0].set_title("AF2-Multimer pae_interaction"); ax[0].legend()
ax[1].set_xlabel("rosetta_dG (REU, more negative better)"); ax[1].set_title("Interface energy (survivors)"); ax[1].legend()
fig.suptitle("BindCraft vs RFdiffusion (EXAMPLE_DATA if mock)")
plt.tight_layout(); plt.savefig("results/p06_headtohead.png", dpi=150); plt.show()
print("saved results/p06_headtohead.png")

saved results/p06_headtohead.png


## 2 · Novelty `[extension]`

Novelty = TM-score of each binder to its nearest natural fold (Foldseek/TM-align; `< 0.5` ≈ novel).
On Colab, compute it per design and compare the two paradigms' novelty distributions. Here we scaffold
the analysis (mock has no real structures), so we just show where it plugs in.

In [27]:
# Scaffold: on Colab, run Foldseek/TM-align on each predicted binder backbone -> tm_to_pdb,
# then compare distributions across paradigms (novel == tm_to_pdb < 0.5).
# Example shape of the analysis once tm_to_pdb is populated:
if "tm_to_pdb" in ranked.columns and ranked["tm_to_pdb"].notna().any():
    for p, g in ranked.groupby("paradigm"):
        novel = (g["tm_to_pdb"] < 0.5).mean()
        print(f"{p:12s}: novel fraction (TM<0.5) = {novel:.2f}")
else:
    print("Novelty scaffold — populate tm_to_pdb with Foldseek/TM-align on Colab, then compare paradigms.")

Novelty scaffold — populate tm_to_pdb with Foldseek/TM-align on Colab, then compare paradigms.


## 3 · Epitope competition vs PD-1 `[extension]`

A binder only **blocks** PD-1 if it covers the PD-1 footprint. `hotspot_overlap` (notebook 02) is our
geometry proxy: the fraction of PD-1-face hotspots the binder contacts. Higher ⇒ more likely a
competitive blocker. Compare the survivors' overlap across paradigms — a strong interface that *misses*
the PD-1 face is not a checkpoint blocker.

In [28]:
bc = pd.read_csv("results/bindcraft_designs.csv")
rf = pd.read_csv("results/rfdiffusion_designs.csv")
pools = pd.concat([bc, rf], ignore_index=True)

# Join overlap onto the ranked survivors.
ov = pools.set_index("design_id")["hotspot_overlap"]
ranked["hotspot_overlap"] = ranked["design_id"].map(ov)
surv = ranked[ranked["layers_passed"] >= 3]

print("epitope-competition overlap of all-layers survivors (SYNTHETIC if mock):")
for p, g in surv.groupby("paradigm"):
    print(f"  {p:12s}: median PD-1-footprint overlap = {g['hotspot_overlap'].median():.2f}  (n={len(g)})")

# "Competitive blockers" = survivors that also cover enough of the PD-1 footprint.
BLOCK_OVERLAP = 0.5
blockers = surv[surv["hotspot_overlap"] >= BLOCK_OVERLAP]
print(f"\nlikely competitive blockers (survivor AND overlap>={BLOCK_OVERLAP}): {len(blockers)}")
print(blockers.groupby("paradigm").size().to_dict())

epitope-competition overlap of all-layers survivors (SYNTHETIC if mock):
  bindcraft   : median PD-1-footprint overlap = 0.67  (n=3)
  rfdiffusion : median PD-1-footprint overlap = 0.67  (n=11)

likely competitive blockers (survivor AND overlap>=0.5): 8
{'bindcraft': 2, 'rfdiffusion': 6}


## 4 · Select the top 10–20 per paradigm

The D★ deliverable wants the **top 10–20 each**. Rank survivors by the composite score and, as a
tie-breaker for *blockers*, prefer higher PD-1-footprint overlap. Save the shortlist for the validation
plan (notebook 05).

In [29]:
top_per = []
for p, g in ranked.groupby("paradigm"):
    g2 = g[g["layers_passed"] >= 3].sort_values(
        ["score", "hotspot_overlap"], ascending=False).head(20)
    top_per.append(g2)
top = pd.concat(top_per, ignore_index=True)
top.to_csv("results/top_candidates.csv", index=False)
print("wrote results/top_candidates.csv:", top.shape, "(top<=20 per paradigm)")
print(top.groupby("paradigm").size().to_dict())
top.head(8)[["design_id", "paradigm", "score", "pae_interaction", "rosetta_dG", "hotspot_overlap"]]

wrote results/top_candidates.csv: (14, 22) (top<=20 per paradigm)
{'bindcraft': 3, 'rfdiffusion': 11}


,design_id,paradigm,score,pae_interaction,rosetta_dG,hotspot_overlap
0,EXAMPLE_DATA_bindcraft_0034,bindcraft,4.2600,9.0,-33.0,0.667
1,EXAMPLE_DATA_bindcraft_0048,bindcraft,3.3367,4.0,-36.0,0.333
2,EXAMPLE_DATA_bindcraft_0041,bindcraft,3.3000,9.0,-30.0,1.000
3,EXAMPLE_DATA_rfdiffusion_0176,rfdiffusion,4.2900,6.0,-37.0,0.667
4,EXAMPLE_DATA_rfdiffusion_0004,rfdiffusion,4.1233,8.0,-37.0,0.333
5,EXAMPLE_DATA_rfdiffusion_0008,rfdiffusion,3.7933,5.0,-43.0,0.333
6,EXAMPLE_DATA_rfdiffusion_0131,rfdiffusion,3.6633,8.0,-39.0,1.000
7,EXAMPLE_DATA_rfdiffusion_0199,rfdiffusion,3.5967,10.0,-34.0,0.333


## D3 (part 2) checklist
- [ ] Head-to-head: hit rate + interface-energy distribution per paradigm (figure `results/p06_headtohead.png`).
- [ ] Novelty compared across paradigms (TM-score to PDB) — or the scaffold wired up on Colab.
- [ ] Epitope-competition vs PD-1: footprint overlap of survivors; "competitive blocker" count.
- [ ] `results/top_candidates.csv`: top 10–20 each, ready for the validation plan.
- [ ] Honest discussion of the two paradigms' different failure modes (not just a winner).

**Next:** `05_validation_plan.ipynb` — the SPR/BLI + PD-1-competition plan.

---

## ▶︎ Section 6 / 6 — `05_validation_plan.ipynb`

---

# 05 · Validation Plan — SPR/BLI + PD-1 competition + controls

**Standard slot:** *validation plan.* **For Project 06 this means:** turn the top candidates into a
**costed, controlled wet-lab plan** — SPR/BLI affinity vs PD-L1, a **PD-1-competition** assay, the
mandatory controls (positive known binder, **scrambled-interface** negative, unrelated negative), an
expression strategy, and the **Boltz-2 affinity** stretch (scaffold only) (D4/D5).

A design that passes every filter is a **hypothesis** — SPR/BLI is what tests it. Needs
`results/top_candidates.csv` (notebook 04).

## Setup paths

In [30]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

paths ready; cwd = /home/user/biofx_python/denovo_protein_design_course/projects/project_06_pdl1_binder/notebooks


## 1 · Draft the experimental validation plan

Generate a plan card from the top candidates: assays, controls, expression, timeline, costed reagents.
Fill the `<...>` from your own numbers; this is the deliverable other people will actually read.

In [31]:
import pandas as pd, os

top = pd.read_csv("results/top_candidates.csv") if os.path.exists("results/top_candidates.csv") else pd.DataFrame()
n_top = len(top)
by_par = top.groupby("paradigm").size().to_dict() if n_top else {}

plan = f"""# PD-L1 Mini-Binder Validation Plan (Project 06 — by <your name>, <date>)

## Candidates
Top {n_top} candidates carried forward ({by_par}); see results/top_candidates.csv.
EVERY in-silico number is a HYPOTHESIS until measured — `pae_interaction` is confidence, not affinity.

## Expression strategy
- Binders: E. coli BL21(DE3), His-tagged, 16-18 C overnight; IMAC + SEC. Small (40-80 aa) -> high yield expected.
- PD-L1 ectodomain (IgV) reagent: mammalian/insect expression or commercial; confirm it is active
  (binds PD-1) before testing binders.

## Assays (go/no-go -> basic -> functional)
1. Go/no-go: express -> SDS-PAGE -> SEC (monodisperse?).
2. Affinity: SPR or BLI vs immobilized PD-L1 -> K_D + kinetics (k_on/k_off). Test a dilution series.
3. Functional (the point): PD-1-COMPETITION assay -> does the binder displace PD-1 from PD-L1?
   (SPR competition, or a cell-based PD-1/PD-L1 blockade reporter assay.)
4. Stability: DSF (Tm). Deep (optional): co-crystal / cryo-EM of the binder-PD-L1 complex.

## Controls (MANDATORY)
- Positive: a known PD-L1 binder (anti-PD-L1 Fab/antibody, or PD-1 ectodomain) -> assay + reagent are active.
- Negative (scrambled-interface): YOUR OWN top design with its interface residues scrambled/mutated
  -> must LOSE binding (cleanest specificity control).
- Negative (unrelated): an unrelated mini-protein of similar size -> should not bind.

## Realistic expectations
In-silico binder hit rates vary widely; the MAJORITY of in-silico hits fail experimentally. Expect
to test many to find a few real binders. Report the experimental hit rate honestly. Do NOT imply a
working binder or fabricate a K_D.

## Timeline + costed reagents (fill in)
- Gene synthesis ({n_top} binders + scrambled-interface negatives): $<...>, <...> weeks (IGSC-screened provider).
- PD-L1 reagent + SPR/BLI chips + anti-PD-L1 positive control: $<...>.
- Personnel/instrument time: <...> weeks.

## Responsible research
Blocking binders to a human checkpoint protein for cancer immunotherapy/diagnostics (in scope).
Gene synthesis via a biosecurity-screening provider; wet lab under institutional biosafety/ethics approval.
"""
os.makedirs("results", exist_ok=True)
open("results/validation_plan.md", "w").write(plan)
print("wrote results/validation_plan.md — fill the <...> placeholders from your numbers.")
print(plan[:600], "...")

wrote results/validation_plan.md — fill the <...> placeholders from your numbers.
# PD-L1 Mini-Binder Validation Plan (Project 06 — by <your name>, <date>)

## Candidates
Top 14 candidates carried forward ({'bindcraft': 3, 'rfdiffusion': 11}); see results/top_candidates.csv.
EVERY in-silico number is a HYPOTHESIS until measured — `pae_interaction` is confidence, not affinity.

## Expression strategy
- Binders: E. coli BL21(DE3), His-tagged, 16-18 C overnight; IMAC + SEC. Small (40-80 aa) -> high yield expected.
- PD-L1 ectodomain (IgV) reagent: mammalian/insect expression or commercial; confirm it is active
  (binds PD-1) before testing binders.

## Assays (go/no-go -> basi ...


## 2 · Build the scrambled-interface negative controls

The single cleanest specificity control: take each top design and **scramble its interface residues**
(the positions contacting PD-L1) — it should **lose** binding. Generating these alongside the real
designs (same expression batch) makes the SPR/BLI comparison airtight. Here we scaffold the
sequence-level scramble deterministically; on Colab, scramble the *interface* positions specifically
using the predicted contacts.

In [32]:
import random
import binder_tools as bt   # bt._hashints gives a DETERMINISTIC seed (Python's hash() is salted)

def scramble_interface(seq, frac=0.4, seed=0):
    """Deterministically shuffle a fraction of the sequence as a NEGATIVE-CONTROL stand-in.
    On Colab, scramble the predicted INTERFACE residues specifically (positions contacting PD-L1)."""
    rng = random.Random(seed)
    seq = list(seq)
    idx = list(range(len(seq)))
    rng.shuffle(idx)
    k = max(1, int(len(seq) * frac))
    chosen = idx[:k]
    vals = [seq[i] for i in chosen]
    rng.shuffle(vals)
    for i, v in zip(chosen, vals):
        seq[i] = v
    return "".join(seq)

negs = []
if n_top and "sequence" in top.columns:
    for _, r in top.iterrows():
        s = str(r.get("sequence", ""))
        if s:
            negs.append(dict(design_id=str(r["design_id"]) + "_SCRAM",
                             parent=r["design_id"], paradigm=r.get("paradigm"),
                             sequence=scramble_interface(s, seed=bt._hashints(r["design_id"]) % 10**6),
                             role="scrambled-interface negative control"))
    pd.DataFrame(negs).to_csv("results/negative_controls.csv", index=False)
    print(f"wrote results/negative_controls.csv: {len(negs)} scrambled-interface negatives")
else:
    print("Run notebook 04 first to produce results/top_candidates.csv with sequences.")

wrote results/negative_controls.csv: 14 scrambled-interface negatives


## 3 · (Stretch) Boltz-2 affinity on top hits `[stretch]`

Boltz-2 can predict a binding-affinity signal for the top complexes. Use it for **relative ranking +
caveats only** — **never fabricate a K_D**, and never present a predicted number as measured. This
tells you which hits to test first, not whether they bind.

In [33]:
# Scaffold ONLY. Do NOT invent affinities. On Colab:
#   pip install boltz; build the (binder, PD-L1) complex input; run boltz predict with affinity mode;
#   read the predicted-affinity signal and report the RELATIVE ranking of the top hits + heavy caveats.
# Pinned upstream (verify): https://github.com/jwohlwend/boltz
print("Boltz-2 affinity is a STRETCH scaffold: relative ranking + caveats only, NEVER a fabricated K_D.")
print("Use it to PRIORITIZE which top hits to test first in SPR/BLI — not as evidence of binding.")

Boltz-2 affinity is a STRETCH scaffold: relative ranking + caveats only, NEVER a fabricated K_D.
Use it to PRIORITIZE which top hits to test first in SPR/BLI — not as evidence of binding.


## D4 / D5 checklist
- [ ] `results/validation_plan.md` completed: SPR/BLI + **PD-1-competition** assay, expression, timeline, costed reagents.
- [ ] Controls specified: positive (known PD-L1 binder), **scrambled-interface** negative (`results/negative_controls.csv`), unrelated negative.
- [ ] (Stretch) Boltz-2 affinity used only for relative ranking, with caveats — no fabricated K_D.
- [ ] Honest framing: every design is a hypothesis until SPR/BLI; report the experimental hit rate.
- [ ] Thesis chapter + 15-min talk + `v1.0` tagged release.

You're done — and this is the **binder-family template** Projects 07–13 and 23 will reuse.